# Church matching: compare feature detectors and matchers

The same task as `churches.ipynb`, but with many methods on the same sample, so you can see which one separates real matches from look-alike churches.

Every method gets the same input (grayscale, long side `RESIZE`) and the same check afterwards (a homography with MAGSAC). Only the detector, the descriptor and the matcher change:

| method | detector + descriptor | matcher |
|---|---|---|
| `orb`, `akaze`, `brisk` | OpenCV, binary descriptors | nearest neighbour (Hamming), ratio test, mutual check |
| `rootsift` | OpenCV SIFT, RootSIFT descriptors | nearest neighbour, ratio test, mutual check |
| `*+nn` | learned (LightGlue package) | the same nearest neighbour matcher |
| `*+lightglue` | learned (LightGlue package) | LightGlue |

The `+nn` and `+lightglue` variants of one detector show what LightGlue itself adds. LightGlue uses context and can align two towers that only look alike; a nearest neighbour matcher cannot.

Two scores per pair:

- `inliers`: homography inliers.
- `cells`: the number of cells of a `GRID` x `GRID` grid that hold an inlier, the lower of the two images. 150 inliers on one patch of brick count as a few cells; matches over the whole facade count as many.

## Parameters

In [ ]:
import os
from pathlib import Path

from common.config import config

PLATE_DIR = Path(config().paths.images.glashelder)
CHURCH_DIR = PLATE_DIR.parent / "onroerend" / "kerken"
EMBEDDINGS = Path(config().paths.embeddings.glashelder).parent
PLATE_STORE = EMBEDDINGS / "glashelder-dino.store"
CHURCH_STORE = EMBEDDINGS / "onroerend-dino.store"

N_CHURCHES = 20
PLATES_PER_CHURCH = 50  # DINOv2 neighbours per church; their union is the plate set
SEED = 0

# pairs you checked by eye, as (church file name, plate file name); always in the sample
TRUE_PAIRS: set[tuple[str, str]] = set()  # e.g. {('114900.jpg', 'fx73v2hq54_tiff.jpg')}

METHODS = [
    "orb",
    "akaze",
    "brisk",
    "rootsift",
    "sift+lightglue",
    "superpoint+nn",
    "superpoint+lightglue",
    "disk+nn",
    "disk+lightglue",
    "aliked+nn",
    "aliked+lightglue",
    "doghardnet+lightglue",
]

RESIZE = 1024  # long side, for every method
MAX_KEYPOINTS = 2048
RATIO = 0.8  # nearest neighbour ratio test
H_PX = 8.0  # homography inlier threshold, in pixels at RESIZE scale
GRID = 8
SCORE = "cells"  # what the rankings sort by: 'cells' or 'inliers'
THREADS = os.cpu_count() or 8

## Setup

In [ ]:
import random
import threading
import time
from collections import Counter
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from lightglue import ALIKED, DISK, SIFT, DoGHardNet, LightGlue, SuperPoint
from lightglue.utils import rbd
from tqdm.auto import tqdm

from common.images import load
from embed.device import get_device
from embed.io import load_embeddings
from embed.search import place, topk

device = get_device()
print(
    torch.__version__, device, torch.cuda.get_device_name() if device == "cuda" else ""
)


def sync():
    if device == "cuda":
        torch.cuda.synchronize()


def name(ref: str) -> str:
    return Path(ref).name

## Sample

Random churches plus the churches of `TRUE_PAIRS`, then the DINOv2 neighbours of each church among the plates, plus the plates of `TRUE_PAIRS`.

In [ ]:
church_store = load_embeddings(CHURCH_STORE, root=CHURCH_DIR)
plate_store = load_embeddings(PLATE_STORE, root=PLATE_DIR)
assert church_store["model_id"] == plate_store["model_id"], (
    "stores use different models"
)

church_row = {name(ref): i for i, ref in enumerate(church_store["images"])}
plate_row = {name(ref): i for i, ref in enumerate(plate_store["images"])}

known = sorted({church_row[c] for c, _ in TRUE_PAIRS})
rest = sorted(set(range(len(church_store["images"]))) - set(known))
rows = known + random.Random(SEED).sample(rest, max(0, N_CHURCHES - len(known)))
churches = [church_store["images"][i] for i in rows]

plate_embeddings = place(plate_store["embeddings"], device, label="plates")
_, neighbours = topk(
    church_store["embeddings"][rows], plate_embeddings, PLATES_PER_CHURCH
)
del plate_embeddings

plate_rows = set(neighbours.flatten().tolist()) | {plate_row[p] for _, p in TRUE_PAIRS}
plates = sorted(plate_store["images"][i] for i in plate_rows)
pairs = [(c, p) for c in churches for p in plates]
print(f"{len(churches)} churches, {len(plates)} plates, {len(pairs):,} pairs")

## Load the images once

Grayscale and resized, so all methods see the same pixels and all keypoints are at the same scale.

In [ ]:
def prepare(ref: str) -> np.ndarray:
    image = np.asarray(load(ref).convert("L"))
    scale = RESIZE / max(image.shape)
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    return cv2.resize(image, None, fx=scale, fy=scale, interpolation=interpolation)


refs = churches + plates
with ThreadPoolExecutor(THREADS) as executor:
    images = dict(
        zip(
            refs,
            tqdm(executor.map(prepare, refs), total=len(refs), desc="load"),
            strict=True,
        )
    )

## Methods

A method is an extractor (image to `Features`) and a matcher (two `Features` to index pairs).

The nearest neighbour matcher runs on the GPU for every descriptor type. Binary descriptors become vectors of +1 and -1, so a dot product gives the Hamming distance. Float descriptors are unit length, so a dot product gives the L2 distance.

In [ ]:
@dataclass
class Features:
    keypoints: np.ndarray  # (N, 2), pixels at RESIZE scale
    descriptors: torch.Tensor  # (N, D) on device: unit length, or +1/-1 for binary
    binary: bool = False
    native: dict | None = None  # what LightGlue takes


@dataclass
class Method:
    extract: Callable[[np.ndarray], Features]
    match: Callable[[Features, Features], np.ndarray]
    threads: bool = False  # extract on CPU threads


def descriptors(desc: np.ndarray, binary: bool, root: bool) -> torch.Tensor:
    if binary:
        bits = np.unpackbits(desc, axis=1).astype(np.float32) * 2 - 1
        return torch.from_numpy(bits).to(device, torch.float16)
    values = desc.astype(np.float32)
    if root:
        values = np.sqrt(values / (np.abs(values).sum(axis=1, keepdims=True) + 1e-8))
    return unit(torch.from_numpy(values).to(device))


def unit(desc: torch.Tensor) -> torch.Tensor:
    return (desc / desc.norm(dim=1, keepdim=True).clamp_min(1e-8)).half()


def opencv(make: Callable, *, binary: bool = False, root: bool = False) -> Callable:
    """An extractor around an OpenCV detector, one detector per thread."""
    local = threading.local()

    def extract(image: np.ndarray) -> Features:
        if not hasattr(local, "detector"):
            local.detector = make()
        found, desc = local.detector.detectAndCompute(image, None)
        if desc is None or len(found) == 0:
            return Features(
                np.zeros((0, 2), np.float32), torch.zeros((0, 1), device=device), binary
            )
        keep = np.argsort([-k.response for k in found])[:MAX_KEYPOINTS]
        points = np.array([found[i].pt for i in keep], dtype=np.float32)
        return Features(points, descriptors(desc[keep], binary, root), binary)

    return extract


def learned(extractor) -> Callable:
    """An extractor around a LightGlue package model, on the already resized image."""

    def extract(image: np.ndarray) -> Features:
        tensor = (
            torch.from_numpy(image).to(device).float().div(255)[None].repeat(3, 1, 1)
        )
        with torch.inference_mode():
            feats = extractor.extract(tensor, resize=None)
        return Features(
            feats["keypoints"][0].cpu().numpy(),
            unit(feats["descriptors"][0]),
            native=feats,
        )

    return extract


NO_MATCHES = np.zeros((0, 2), dtype=np.int64)


def nearest(f0: Features, f1: Features) -> np.ndarray:
    """Mutual nearest neighbours that pass the ratio test."""
    if len(f0.keypoints) < 2 or len(f1.keypoints) < 2:
        return NO_MATCHES
    dot = (f0.descriptors @ f1.descriptors.T).float()
    if f0.binary:
        distance = (f0.descriptors.shape[1] - dot) / 2
    else:
        distance = (2 - 2 * dot).clamp_min(0).sqrt()
    best, nn = distance.topk(2, dim=1, largest=False)
    passes = best[:, 0] < RATIO * best[:, 1]
    mutual = distance.argmin(dim=0)[nn[:, 0]] == torch.arange(
        len(nn), device=distance.device
    )
    kept = torch.nonzero(passes & mutual).squeeze(1)
    return torch.stack([kept, nn[kept, 0]], dim=1).cpu().numpy()


def lightglue(features: str) -> Callable:
    matcher = LightGlue(features=features).eval().to(device)

    def match(f0: Features, f1: Features) -> np.ndarray:
        if len(f0.keypoints) < 2 or len(f1.keypoints) < 2:
            return NO_MATCHES
        with torch.inference_mode():
            out = rbd(matcher({"image0": f0.native, "image1": f1.native}))
        return out["matches"].cpu().numpy()

    return match


def model(cls):
    return cls(max_num_keypoints=MAX_KEYPOINTS).eval().to(device)


# factories, so a method only loads its models when it runs
FACTORIES: dict[str, Callable[[], Method]] = {
    "orb": lambda: Method(
        opencv(lambda: cv2.ORB_create(MAX_KEYPOINTS), binary=True),
        nearest,
        threads=True,
    ),
    "akaze": lambda: Method(
        opencv(cv2.AKAZE_create, binary=True), nearest, threads=True
    ),
    "brisk": lambda: Method(
        opencv(cv2.BRISK_create, binary=True), nearest, threads=True
    ),
    "rootsift": lambda: Method(
        opencv(lambda: cv2.SIFT_create(MAX_KEYPOINTS), root=True), nearest, threads=True
    ),
    "sift+lightglue": lambda: Method(learned(model(SIFT)), lightglue("sift")),
    "superpoint+nn": lambda: Method(learned(model(SuperPoint)), nearest),
    "superpoint+lightglue": lambda: Method(
        learned(model(SuperPoint)), lightglue("superpoint")
    ),
    "disk+nn": lambda: Method(learned(model(DISK)), nearest),
    "disk+lightglue": lambda: Method(learned(model(DISK)), lightglue("disk")),
    "aliked+nn": lambda: Method(learned(model(ALIKED)), nearest),
    "aliked+lightglue": lambda: Method(learned(model(ALIKED)), lightglue("aliked")),
    "doghardnet+lightglue": lambda: Method(
        learned(model(DoGHardNet)), lightglue("doghardnet")
    ),
}

## Check and score

The same for every method. Only the inlier points are kept, for the plots.

In [ ]:
def inliers(pts0: np.ndarray, pts1: np.ndarray) -> np.ndarray:
    if len(pts0) < 8:
        return np.zeros(len(pts0), dtype=bool)
    homography, mask = cv2.findHomography(
        pts0, pts1, cv2.USAC_MAGSAC, H_PX, maxIters=10_000, confidence=0.999
    )
    if homography is None:
        return np.zeros(len(pts0), dtype=bool)
    return mask.ravel().astype(bool)


def cells(points: np.ndarray, shape: tuple[int, ...]) -> int:
    height, width = shape[:2]
    x = np.minimum((points[:, 0] / width * GRID).astype(int), GRID - 1)
    y = np.minimum((points[:, 1] / height * GRID).astype(int), GRID - 1)
    return len(np.unique(y * GRID + x))


def score(method: Method, feats: dict[str, Features], church: str, plate: str) -> dict:
    f0, f1 = feats[church], feats[plate]
    matched = method.match(f0, f1)
    pts0, pts1 = f0.keypoints[matched[:, 0]], f1.keypoints[matched[:, 1]]
    mask = inliers(pts0, pts1)
    pts0, pts1 = pts0[mask], pts1[mask]
    return {
        "church": church,
        "plate": plate,
        "matches": len(matched),
        "inliers": len(pts0),
        "cells": min(
            cells(pts0, images[church].shape), cells(pts1, images[plate].shape)
        ),
        "pts0": pts0,
        "pts1": pts1,
    }

## Run

One method at a time, so only one set of features is in memory. A method that fails (for example a missing weight download) is skipped and reported.

In [ ]:
def run(method_name: str) -> dict:
    method = FACTORIES[method_name]()
    method.extract(images[refs[0]])  # warm up

    start = time.perf_counter()
    if method.threads:
        with ThreadPoolExecutor(THREADS) as executor:
            found = list(executor.map(method.extract, (images[r] for r in refs)))
    else:
        found = [
            method.extract(images[r])
            for r in tqdm(refs, desc=f"{method_name} extract", leave=False)
        ]
    sync()
    extract_s = (time.perf_counter() - start) / len(refs)
    feats = dict(zip(refs, found, strict=True))

    start = time.perf_counter()
    scored = [
        score(method, feats, c, p)
        for c, p in tqdm(pairs, desc=f"{method_name} match", leave=False)
    ]
    match_s = (time.perf_counter() - start) / len(pairs)

    return {
        "extract_ms": extract_s * 1e3,
        "match_ms": match_s * 1e3,
        "keypoints": float(np.median([len(f.keypoints) for f in found])),
        "pairs": scored,
    }


results: dict[str, dict] = {}
for method_name in METHODS:
    try:
        results[method_name] = run(method_name)
        r = results[method_name]
        print(
            f"{method_name:>22}: {r['extract_ms']:6.1f} ms/image, {r['match_ms']:6.1f} ms/pair"
        )
    except Exception as error:
        print(f"{method_name:>22}: skipped, {type(error).__name__}: {error}")
    finally:
        if device == "cuda":
            torch.cuda.empty_cache()

## Summary

Without labels, the best signal is how the top of each ranking looks:

- `churches@30`: how many different churches are in the 30 best pairs. Few means one or two look-alike churches take the whole top.
- `hub`: the church that has the most of the 30 best pairs, and how many it has.

With `TRUE_PAIRS` set, the last columns show for every true pair where the plate stands among the plates of its church, and where the church stands among the churches for that plate (1 is best).

In [ ]:
TOP = 30


def ranked(rows: list[dict]) -> list[dict]:
    return sorted(rows, key=lambda r: (r[SCORE], r["inliers"]), reverse=True)


def rank(rows: list[dict], key: str, value: str, other: str, target: str) -> int:
    """Where `target` stands among the rows that share `value` in `key`."""
    group = ranked([r for r in rows if name(r[key]) == value])
    return next(i for i, r in enumerate(group, 1) if name(r[other]) == target)


header = f"{'method':>22} {'ms/img':>7} {'ms/pair':>8} {'kpts':>6} {'churches@30':>12} {'hub':>16}"
print(header + "".join(f"  {c[:8]}/{p[:8]}" for c, p in sorted(TRUE_PAIRS)))
for method_name, r in results.items():
    top = Counter(name(x["church"]) for x in ranked(r["pairs"])[:TOP])
    hub, hub_count = top.most_common(1)[0]
    ranks = "".join(
        f"  {rank(r['pairs'], 'church', c, 'plate', p):>8}/{rank(r['pairs'], 'plate', p, 'church', c):<8}"
        for c, p in sorted(TRUE_PAIRS)
    )
    print(
        f"{method_name:>22} {r['extract_ms']:>7.1f} {r['match_ms']:>8.1f} {r['keypoints']:>6.0f}"
        f" {len(top):>12} {hub:>12} x{hub_count:<3}{ranks}"
    )

## Score distributions

A good method has most pairs near zero and a few clearly apart on the right.

In [ ]:
columns = 4
lines = -(-len(results) // columns)
fig, axes = plt.subplots(
    lines, columns, figsize=(4 * columns, 2.6 * lines), squeeze=False
)
for ax, (method_name, r) in zip(axes.flat, results.items(), strict=False):
    values = np.array([x[SCORE] for x in r["pairs"]])
    ax.hist(values, bins=np.arange(0, values.max() + 2))
    ax.set_yscale("log")
    ax.set_title(method_name, fontsize=9)
for ax in axes.flat[len(results) :]:
    ax.axis("off")
fig.supxlabel(SCORE)
plt.tight_layout()
plt.show()

## Best pairs per method

One row per method, its `SHOW` best pairs, inliers only. This is where you see which method finds real matches.

In [ ]:
SHOW = 4


def pair_image(row: dict) -> np.ndarray:
    left, right = images[row["church"]], images[row["plate"]]
    canvas = np.full(
        (max(len(left), len(right)), left.shape[1] + right.shape[1], 3), 255, np.uint8
    )
    canvas[: len(left), : left.shape[1]] = left[..., None]
    canvas[: len(right), left.shape[1] :] = right[..., None]
    offset = np.array([left.shape[1], 0], dtype=np.float32)
    segments = (
        np.stack([row["pts0"], row["pts1"] + offset], axis=1).round().astype(np.int32)
    )
    cv2.polylines(canvas, list(segments), False, (0, 200, 0), 1, cv2.LINE_AA)
    return canvas


def label(row: dict) -> str:
    return f"{name(row['church'])} / {name(row['plate'])}\n{row['inliers']} inliers, {row['cells']} cells"


fig, axes = plt.subplots(
    len(results), SHOW, figsize=(5 * SHOW, 3 * len(results)), squeeze=False
)
for line, (method_name, r) in zip(axes, results.items(), strict=True):
    for ax, row in zip(line, ranked(r["pairs"])[:SHOW], strict=False):
        ax.imshow(pair_image(row))
        ax.set_title(label(row), fontsize=7)
    for ax in line:
        ax.set_xticks([])
        ax.set_yticks([])
    line[0].set_ylabel(method_name, fontsize=9)
plt.tight_layout()
plt.show()

## One pair, every method

Set `PAIR` to a (church, plate) file name pair, for example one from `TRUE_PAIRS`, to see how each method handles it.

In [ ]:
PAIR = next(iter(sorted(TRUE_PAIRS)), None)  # or ('181724.jpg', '9g5gb5116v_tiff.jpg')

if PAIR is None:
    print("set PAIR to a (church, plate) file name pair")
else:
    found = {
        method_name: next(
            x for x in r["pairs"] if (name(x["church"]), name(x["plate"])) == PAIR
        )
        for method_name, r in results.items()
    }
    columns = 3
    lines = -(-len(found) // columns)
    fig, axes = plt.subplots(
        lines, columns, figsize=(6 * columns, 3.5 * lines), squeeze=False
    )
    for ax, (method_name, row) in zip(axes.flat, found.items(), strict=False):
        ax.imshow(pair_image(row))
        ax.set_title(
            f"{method_name}: {row['inliers']} inliers, {row['cells']} cells", fontsize=8
        )
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes.flat[len(found) :]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()